In [1]:
from pyspark.sql import SparkSession
import os
import pandas as pd
import time
import string
import pathlib
import random
import threading
import time
from urllib.parse import urlsplit, urlunsplit
import requests
import json
from py4j.protocol import Py4JJavaError, Py4JError
import glob
import psutil

In [2]:
# Global configuration
SPARK_MEMORY = 10
SPARK_CORES = 4
DBHOST = 'postgres'
QUERY_TIMEOUT = 60 * 30
QUERY_TIMEOUT = 60 * 180

In [3]:
def create_spark():
    spark = SparkSession.builder \
        .appName("app") \
        .master(f'local[{SPARK_CORES}]') \
        .config("spark.driver.memory", f'{SPARK_MEMORY}g') \
        .config("spark.executor.memory", f'{SPARK_MEMORY}g') \
        .config("spark.memory.offHeap.enabled",False) \
        .config("spark.jars", "postgresql-42.3.3.jar") \
        .getOrCreate()
    return spark

## Cluster
# def create_spark():
#     spark = SparkSession.builder \
#         .appName("app") \
#         .master('spark://10.100.42.35:7078') \
#         .config("spark.driver.memory", f'{SPARK_MEMORY}g') \
#         .config("spark.executor.memory", f'{SPARK_MEMORY}g') \
#         .config("spark.driver.host", "10.100.42.223") \
#         .config("spark.driver.bindAddress", "0.0.0.0") \
#         .config("spark.driver.port", "4060") \
#         .config("spark.memory.offHeap.enabled",OFFHEAP) \
#         .config("spark.jars", "postgresql-42.3.3.jar") \
#         .getOrCreate()
#     return spark

In [4]:
def extract_metrics(spark, group_id):
    parsed = list(urlsplit(spark.sparkContext.uiWebUrl))
    host_port = parsed[1]
    parsed[1] = 'localhost' + host_port[host_port.find(':'):]
    API_URL = f'{urlunsplit(parsed)}/api/v1'

    app_id = spark.sparkContext.applicationId
    sql_queries = requests.get(API_URL + f'/applications/{app_id}/sql', params={'length': '100000'}).json()
    query_ids = [q['id'] for q in sql_queries if q['description'] == group_id]
    if (len(query_ids) == 0):
        print(f'query with group {group_id} not found')
        return None
    query_id = query_ids[0]
    print(f'query id: {query_id}')
    
    query_details = requests.get(API_URL + f'/applications/{app_id}/sql/{query_id}',
                                 params={'details': 'true', 'planDescription': 'true'}).json()
    
    success_job_ids = query_details['successJobIds']
    running_job_ids = query_details['runningJobIds']
    failed_job_ids = query_details['failedJobIds']
    
    job_ids = success_job_ids + running_job_ids + failed_job_ids
    
    job_details = [requests.get(API_URL + f'/applications/{app_id}/jobs/{jid}').json() for jid in job_ids]
    
    job_stages = {}
    
    for j in job_details:
        stage_ids = j['stageIds']
        
        stage_params = {'details': 'true', 'withSummaries': 'true'}
        stages = [requests.get(API_URL + f'/applications/{app_id}/stages/{sid}', stage_params) for sid in stage_ids]
        
        job_stages[j['jobId']] = [stage.json() for stage in stages if stage.status_code == 200] # can be 404
    
    return query_details, job_details, job_stages

In [5]:
def import_db(spark, dbname):
    
    username = dbname
    password = dbname
    dbname = dbname

    df_tables = spark.read.format("jdbc") \
    .option("url", f'jdbc:postgresql://{DBHOST}:5432/{dbname}') \
    .option("driver", "org.postgresql.Driver") \
    .option("dbtable", "information_schema.tables") \
    .option("user", username) \
    .option("password", password) \
    .load()

    for idx, row in df_tables.toPandas().iterrows():
        if row.table_schema == 'public':
            table_name = row.table_name
            df = spark.read.format("jdbc") \
                .option("url", f'jdbc:postgresql://{DBHOST}:5432/{dbname}') \
                .option("driver", "org.postgresql.Driver") \
                .option("dbtable", table_name) \
                .option("user", username) \
                .option("password", password) \
                .load()
    
            print(table_name)
            #print(df.show())
            df.createOrReplaceTempView(table_name)

def random_str(size=16, chars=string.ascii_uppercase + string.digits):
    return ''.join(random.choice(chars) for _ in range(size))

def set_group_id(spark):
    group_id = random_str()
    spark.sparkContext.setJobGroup(group_id, group_id)
    return group_id

def cancel_query(spark, seconds, group_id):
    time.sleep(seconds)
    print("cancelling jobs with id " + group_id)
    print(spark.sparkContext.cancelJobGroup(group_id))
    print("cancelled job")

def cancel_query_after(spark, seconds):
    group_id = random_str()
    spark.sparkContext.setJobGroup(group_id, group_id)
    threading.Thread(target=cancel_query, args=(spark, seconds, group_id,)).start()
    return group_id
    
def run_query(spark, file):
    with open(file, 'r') as f:
        query = '\n'.join(filter(lambda line: not line.startswith('limit') and not line.startswith('-'), f.readlines()))
        
        print("running query: \n" + query)
        return spark.sql(query)

def get_resource_usage(t):
    return {
        'time': t,
        'memory': psutil.virtual_memory(),
        'cpu': psutil.cpu_percent(interval=None, percpu=True),
        'cpu_total': psutil.cpu_percent(interval=None, percpu=False)
    }
def explain_str(df):
    return df._sc._jvm.PythonSQLUtils.explainString(df._jdf.queryExecution(), 'extended')

In [6]:
resource_usage = []

def measure_resource_usage(resource_usage):
    t = threading.current_thread()
    secs = 0
    while getattr(t, "do_run", True):
        resource_usage.append(get_resource_usage(secs))
        #print("resource usage: " + str(resource_usage))
        secs += 1
        time.sleep(1)

def benchmark_query(spark, query, respath, run):
    spark.sparkContext._jvm.System.gc()
    start_time = time.time()

    resource_usage = []

    measure_thread = threading.Thread(target=measure_resource_usage, args=(resource_usage, ))
    measure_thread.start()

    group_id = cancel_query_after(spark, QUERY_TIMEOUT)
    df1 = run_query(spark, query)
    df1.show()

    measure_thread.do_run = False

    end_time = time.time()
    diff_time = end_time - start_time

    execution, jobs, job_stages = extract_metrics(spark, group_id)

    with open(respath + f'/resource-usage-{run}.json', 'w') as f:
        f.write(json.dumps(resource_usage, indent=2))
    with open(respath + f'/explain-{run}.txt', 'w') as f:
        f.write(explain_str(df1))

    resource_list = map(lambda r: [r['time'], r['memory'].used, r['cpu_total']], resource_usage)
    resource_df = pd.DataFrame(resource_list, columns = ['time', 'memory_used', 'cpu_used'])
    resource_df.to_csv(respath + f'/resource-usage-{run}.csv')

    peak_memory = max(map(lambda r: r['memory'].used, resource_usage)) / (1000 * 1000 * 1000) # GB

    if execution is not None:
            with open(respath + f'/execution-{run}.json', 'w') as f:
                f.write(json.dumps(execution, indent=2))
            with open(respath + f'/jobs-{run}.json', 'w') as f:
                f.write(json.dumps(jobs, indent=2))
            with open(respath + f'/stages-{run}.json', 'w') as f:
                f.write(json.dumps(job_stages, indent=2))
    return (diff_time, peak_memory)

def benchmark(spark, dbname, query_file, mode, run, syn = False):
    #spark.sql("SET spark.sql.yannakakis.enabled = false").show()
    # run the query once to warm up Spark (load the relation in memory)
    #df0 = run_query(query)
    #df0.show()
    
    query_name = os.path.basename(query_file)

    respath = f'benchmark-results-{dbname}/' + query_name + "/" + mode
    if(syn):
        respath = f'benchmark-results-{dbname}-syn/' + query_name + "/" + mode
    pathlib.Path(respath).mkdir(parents=True, exist_ok=True)

    if mode == "opt":
        spark.sql("SET spark.sql.yannakakis.enabled = true").show()
    elif mode == "ref":
        spark.sql("SET spark.sql.yannakakis.enabled = false").show()
    else:
        return []

    try:
        (runtime, peak_memory) = benchmark_query(spark, query_file, respath, run)
        return [query_name, runtime, peak_memory, mode, run]
    except Py4JError as e:
        print('timeout or error: ' + str(e))
        return [query_name, None, None, mode, run]

def benchmark_all(dbname, mode, runs, queries, group_in_leaves=False, physical_cj=False):
    spark = create_spark()
    import_db(spark, dbname)

    if physical_cj:
        spark.sql("SET spark.sql.codegen.wholeStage = true").show()
        spark.sql("SET spark.sql.yannakakis.physicalCountJoinEnabled = true").show()
    else:
        spark.sql("SET spark.sql.codegen.wholeStage = true").show()
        spark.sql("SET spark.sql.yannakakis.physicalCountJoinEnabled = false").show()
    if group_in_leaves:
        spark.sql("SET spark.sql.yannakakis.countGroupInLeaves = true").show()
    else:
        spark.sql("SET spark.sql.yannakakis.countGroupInLeaves = false").show()

    results_df = df = pd.DataFrame([], columns = ['query', 'runtime', 'peak_memory', 'mode', 'run', 'group_leaves', 'physical_cj'])
    results_file = f'benchmark-results-{dbname}/results-{mode}.csv'
    if (os.path.exists(results_file)):
        results_df = pd.read_csv(results_file, index_col=0)

    for run in runs:
        for q in queries:
            results = [benchmark(spark, dbname, q, mode, run) + [group_in_leaves, physical_cj]]
            new_df = pd.DataFrame(results, columns = ['query', 'runtime', 'peak_memory', 'mode', 'run', 'group_leaves', 'physical_cj'])
            results_df = pd.concat([results_df, new_df], ignore_index=True)
            results_df.to_csv(f'benchmark-results-{dbname}/results-{mode}.csv')
            print(results_df)
    

In [7]:
###new def benchmark from alex december

def benchmark_all(dbname, mode, runs, queries, group_in_leaves=False, physical_cj=False, enable_unguarded=False, syn = False):
    spark = create_spark()
    import_db(spark, dbname)

    if physical_cj:
        spark.sql("SET spark.sql.codegen.wholeStage = true").show()
        spark.sql("SET spark.sql.yannakakis.physicalCountJoinEnabled = true").show()
    else:
        spark.sql("SET spark.sql.codegen.wholeStage = true").show()
        spark.sql("SET spark.sql.yannakakis.physicalCountJoinEnabled = false").show()
    if group_in_leaves:
        spark.sql("SET spark.sql.yannakakis.countGroupInLeaves = true").show()
    else:
        spark.sql("SET spark.sql.yannakakis.countGroupInLeaves = false").show()
    if enable_unguarded:
        spark.sql("SET spark.sql.yannakakis.unguardedEnabled = true").show()
    else:
        spark.sql("SET spark.sql.yannakakis.unguardedEnabled = false").show()

    results_df = df = pd.DataFrame([], columns = ['query', 'runtime', 'peak_memory', 'mode', 'run', 'group_leaves', 'physical_cj'])
    results_file = f'benchmark-results-{dbname}/results-{mode}.csv'
    if (syn):
        results_file = f'benchmark-results-{dbname}-syn/results-{mode}.csv'
    else:
        print("Folder not found.")
    
    if (os.path.exists(results_file)):
        results_df = pd.read_csv(results_file, index_col=0)

    for run in runs:
        for q in queries:
            print(q)
            results = [benchmark(spark, dbname, q, mode, run, syn) + [group_in_leaves, physical_cj]]
            new_df = pd.DataFrame(results, columns = ['query', 'runtime', 'peak_memory', 'mode', 'run', 'group_leaves', 'physical_cj'])
            results_df = pd.concat([results_df, new_df], ignore_index=True)
            results_df.to_csv(results_file)
            print(results_df)

## SNAP Benchmark

### Optimized execution

In [ ]:
#### benchmark configuration
group_in_leaves = False
dbname = 'snap'
mode = 'opt'
runs = ['1', '2', '3', '4', '5', '6']
#runs = ['1']
####

tables = ['patents', 'wiki', 'google', 'dblp']
#tables = ['wiki']


for tablename in tables:
    queries = sorted(glob.glob(f'snap-queries/all/{tablename}-*'))
    print('running queries: ' + str(queries))
    benchmark_all(dbname, mode, runs, queries, physical_cj=True)



### Ref execution

In [ ]:
#### benchmark configuration
group_in_leaves = False
dbname = 'snap'
mode = 'ref'
runs = ['1', '2', '3', '4', '5', '6']
####

queries = ['snap-queries/all/patents-path02.sql',
          'snap-queries/all/patents-path03.sql',
          'snap-queries/all/patents-path04.sql',
          'snap-queries/all/patents-path05.sql',
          'snap-queries/all/patents-tree01.sql',
          'snap-queries/all/wiki-path02.sql',
           'snap-queries/all/google-path02.sql',
           'snap-queries/all/google-path03.sql',
           'snap-queries/all/google-path04.sql',
           'snap-queries/all/dblp-path02.sql',
           'snap-queries/all/dblp-path03.sql',
           'snap-queries/all/dblp-path04.sql',
           'snap-queries/all/dblp-path05.sql',
           'snap-queries/all/dblp-tree01.sql',
           'snap-queries/all/dblp-tree02.sql'
          ]


print('running queries: ' + str(queries))

benchmark_all(dbname, mode, runs, queries)

## LSQB Benchmark

In [ ]:
#### benchmark configuration
dbname = 'lsqb'
group_in_leaves = False
physical_cj = True
#mode = 'opt'
runs = ['1', '2', '3', '4', '5', '6']
runs = ['1', '2']
####

queries = ['lsqb/sql/q1.sql', 'lsqb/sql/q4.sql']
queries_hints = ['lsqb/sql/q1-hint.sql', 'lsqb/sql/q4-hint.sql']

print('running queries: ' + str(queries))
#benchmark_all(dbname, 'opt', runs, queries, group_in_leaves=False, physical_cj=True)
#benchmark_all(dbname, 'opt', runs, queries_hints, group_in_leaves=False, physical_cj=True)

#benchmark_all(dbname, 'opt', ['1'], ['lsqb/sql/q4.sql', 'lsqb/sql/q4-hint.sql'], group_in_leaves=False, physical_cj=True)
#benchmark_all(dbname, 'opt', ['3', '4', '5', '6'], ['lsqb/sql/q4.sql', 'lsqb/sql/q4-hint.sql'], group_in_leaves=False, physical_cj=True)
#benchmark_all(dbname, 'opt', ['1', '2', '3', '4', '5', '6'], ['lsqb/sql/q4.sql', 'lsqb/sql/q4-hint.sql'], group_in_leaves=False, physical_cj=False)

#benchmark_all(dbname, 'opt', ['1', '2', '3', '4', '5', '6'], ['lsqb/sql/q1-hint.sql'], group_in_leaves=False, physical_cj=False)
#benchmark_all(dbname, 'opt', ['1', '2', '3', '4', '5', '6'], ['lsqb/sql/q1-hint.sql'], group_in_leaves=False, physical_cj=True)
#benchmark_all(dbname, 'opt', ['3', '4', '5', '6'], ['lsqb/sql/q1.sql'], group_in_leaves=False, physical_cj=False)
#benchmark_all(dbname, 'opt', ['3', '4', '5', '6'], ['lsqb/sql/q1.sql'], group_in_leaves=False, physical_cj=True)
benchmark_all(dbname, 'ref', ['4', '5', '6'], ['lsqb/sql/q1.sql'])


#benchmark_all(dbname, 'opt', runs, queries_hints, group_in_leaves=False, physical_cj=True)
#benchmark_all(dbname, 'ref', ['3', '4', '5', '6'], ['lsqb/sql/q4.sql'])

## TPC-H Benchmark

In [13]:
#### benchmark configuration
group_in_leaves = False
dbname = 'tpch'
runs = ['1', '2', '3', '4', '5', '6']
#runs = ['x1', 'x2']
####

queries = [
           'tpch-kit/dbgen/queries/postgres/2.sql',
          'tpch-kit/dbgen/queries/postgres/3.sql',
           'tpch-kit/dbgen/queries/postgres/4.sql',
          'tpch-kit/dbgen/queries/postgres/5.sql',
           'tpch-kit/dbgen/queries/postgres/6.sql',
          'tpch-kit/dbgen/queries/postgres/7.sql',
           'tpch-kit/dbgen/queries/postgres/8.sql',
          'tpch-kit/dbgen/queries/postgres/9.sql',
           'tpch-kit/dbgen/queries/postgres/10.sql',
          'tpch-kit/dbgen/queries/postgres/11.sql',
           'tpch-kit/dbgen/queries/postgres/12.sql',
          'tpch-kit/dbgen/queries/postgres/13.sql',
           'tpch-kit/dbgen/queries/postgres/14.sql'
           ]

queries = ['tpch-kit/dbgen/queries/postgres/16.sql',
          'tpch-kit/dbgen/queries/postgres/17.sql',
           'tpch-kit/dbgen/queries/postgres/18.sql',
          'tpch-kit/dbgen/queries/postgres/19.sql',
           'tpch-kit/dbgen/queries/postgres/20.sql',
          'tpch-kit/dbgen/queries/postgres/21.sql',
           'tpch-kit/dbgen/queries/postgres/22.sql']
queries = ['tpch-kit/dbgen/queries/postgres/3.sql']
#queries = ['tpch-queries/2-subq.sql'] #, 'tpch-queries/2-subq-hint.sql']

print('running queries: ' + str(queries))
benchmark_all(dbname, 'opt', ['1'], queries, physical_cj=True)
#benchmark_all(dbname, 'opt', ['3', '4', '5', '6'], queries, group_in_leaves = group_in_leaves, physical_cj=True)
#benchmark_all(dbname, 'opt', ['1', '2', '3', '4', '5', '6'], queries, group_in_leaves = group_in_leaves, physical_cj=False)
#benchmark_all(dbname, 'opt', runs, queries, group_in_leaves = group_in_leaves, physical_cj=False)


running queries: ['tpch-kit/dbgen/queries/postgres/3.sql']
supplier
partsupp
lineitem


part
customer
orders
nation
region
+--------------------+-----+
|                 key|value|
+--------------------+-----+
|spark.sql.codegen...| true|
+--------------------+-----+

+--------------------+-----+
|                 key|value|
+--------------------+-----+
|spark.sql.yannaka...| true|
+--------------------+-----+

+--------------------+-----+
|                 key|value|
+--------------------+-----+
|spark.sql.yannaka...|false|
+--------------------+-----+

+--------------------+-----+
|                 key|value|
+--------------------+-----+
|spark.sql.yannaka...|false|
+--------------------+-----+

Folder not found.
tpch-kit/dbgen/queries/postgres/3.sql
+--------------------+-----+
|                 key|value|
+--------------------+-----+
|spark.sql.yannaka...| true|
+--------------------+-----+

running query: 




select

	l_orderkey,

	sum(l_extendedprice * (1 - l_discount)) as revenue,

	o_orderdate,

	o_shippriority

from

	customer,

	orders,

	lineitem

where

	c_mk

25/03/28 12:27:46 WARN RewriteJoinsAsSemijoins: applying rewriting to join: Aggregate [l_orderkey#3223L, o_orderdate#3302, o_shippriority#3305], [l_orderkey#3223L, sum((l_extendedprice#3228 * (1 - l_discount#3229))) AS revenue#3430, o_orderdate#3302, o_shippriority#3305]
+- Project [o_orderdate#3302, o_shippriority#3305, l_orderkey#3223L, l_extendedprice#3228, l_discount#3229]
   +- Join Inner, (l_orderkey#3223L = cast(o_orderkey#3298 as bigint))
      :- Project [o_orderkey#3298, o_orderdate#3302, o_shippriority#3305]
      :  +- Join Inner, (cast(c_custkey#3280 as bigint) = o_custkey#3299L)
      :     :- Project [c_custkey#3280]
      :     :  +- Filter ((staticinvoke(class org.apache.spark.sql.catalyst.util.CharVarcharCodegenUtils, StringType, readSidePadding, c_mktsegment#3286, 10, true, false, true) = BUILDING  ) AND isnotnull(c_custkey#3280))
      :     :     +- Relation [c_custkey#3280,c_name#3281,c_address#3282,c_nationkey#3283L,c_phone#3284,c_acctbal#3285,c_mktsegment#3286,c

timeout or error: An error occurred while calling o797.showString.
: java.lang.IllegalStateException: Couldn't find o_custkey#3299L in [l_orderkey#3223L,o_orderdate#3302,o_shippriority#3305,sum((l_extendedprice#3228 * (1 - l_discount#3229)))#3431]
	at org.apache.spark.sql.catalyst.expressions.BindReferences$$anonfun$bindReference$1.applyOrElse(BoundAttribute.scala:80)
	at org.apache.spark.sql.catalyst.expressions.BindReferences$$anonfun$bindReference$1.applyOrElse(BoundAttribute.scala:73)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$transformDownWithPruning$1(TreeNode.scala:461)
	at org.apache.spark.sql.catalyst.trees.CurrentOrigin$.withOrigin(origin.scala:76)
	at org.apache.spark.sql.catalyst.trees.TreeNode.transformDownWithPruning(TreeNode.scala:461)
	at org.apache.spark.sql.catalyst.trees.TreeNode.transformDown(TreeNode.scala:437)
	at org.apache.spark.sql.catalyst.trees.TreeNode.transform(TreeNode.scala:405)
	at org.apache.spark.sql.catalyst.expressions.BindReferences$.

/tmp/ipykernel_22/2436655581.py:37: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, new_df], ignore_index=True)


## TPC-H Syn

In [ ]:
#### benchmark configuration syn tpc h
group_in_leaves = False
dbname = 'tpch'
runs = ['1', '2', '3', '4', '5', '6']
#1,3, 5, 4, 14, 18, 19, 23, 30, 31, 33, 37 are countjoin


queries = ['syn-tpc-h/q2.sql',
           'syn-tpc-h/q6.sql',
           'syn-tpc-h/q7.sql',
           'syn-tpc-h/q8.sql',
           'syn-tpc-h/q9.sql',
           'syn-tpc-h/q10.sql',
           'syn-tpc-h/q11.sql',
           'syn-tpc-h/q12.sql',
           'syn-tpc-h/q13.sql',
           'syn-tpc-h/q15.sql',
           'syn-tpc-h/q16.sql',
           'syn-tpc-h/q17.sql',
           'syn-tpc-h/q20.sql',
           'syn-tpc-h/q21.sql',
           'syn-tpc-h/q22.sql',
           'syn-tpc-h/q24.sql',
           'syn-tpc-h/q25.sql'
          ]

queries = ['syn-tpc-h/q32.sql',
           'syn-tpc-h/q34.sql',
           'syn-tpc-h/q35.sql',
           'syn-tpc-h/q36.sql',
           'syn-tpc-h/q38.sql',
           'syn-tpc-h/q39.sql']

queries = ['syn-tpc-h/q41.sql',
          'syn-tpc-h/q42.sql',
          'syn-tpc-h/q43.sql',
          'syn-tpc-h/q44.sql',
          'syn-tpc-h/q45.sql']


queries = ['syn-tpc-h/q23.sql']


print('running queries: ' + str(queries))
benchmark_all(dbname, 'opt', ['1'], queries, physical_cj=True, syn = True)
#benchmark_all(dbname, 'ref', ['1'], queries, physical_cj=False, syn = True)


## TPC-DS Syn

In [13]:
#### benchmark configuration syn tpc ds
group_in_leaves = False
dbname = 'tpcds'
runs = ['1', '2', '3', '4', '5', '6']


queries = ['syn-tpc-ds/q1.sql',
           'syn-tpc-ds/q2.sql',
           'syn-tpc-ds/q3.sql']


print('running queries: ' + str(queries))
benchmark_all(dbname, 'opt', ['1'], queries, physical_cj=True)

running queries: ['syn-tpc-ds/q1.sql', 'syn-tpc-ds/q2.sql', 'syn-tpc-ds/q3.sql']
dbgen_version
customer_address
customer_demographics
date_dim
warehouse
ship_mode
time_dim
reason
income_band
item
store
call_center


25/03/05 10:33:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


customer
web_site
store_returns
household_demographics
web_page
promotion
catalog_page
inventory
catalog_returns
web_returns
web_sales
catalog_sales
store_sales
+--------------------+-----+
|                 key|value|
+--------------------+-----+
|spark.sql.codegen...| true|
+--------------------+-----+

+--------------------+-----+
|                 key|value|
+--------------------+-----+
|spark.sql.yannaka...| true|
+--------------------+-----+

+--------------------+-----+
|                 key|value|
+--------------------+-----+
|spark.sql.yannaka...|false|
+--------------------+-----+

+--------------------+-----+
|                 key|value|
+--------------------+-----+
|spark.sql.yannaka...|false|
+--------------------+-----+

+--------------------+-----+
|                 key|value|
+--------------------+-----+
|spark.sql.yannaka...| true|
+--------------------+-----+

running query: 
SELECT

    s_store_name AS Store,

    SUM(ss_ext_sales_price) AS Total_Sales,

    AVG(ss_e

25/03/05 10:33:04 WARN RewriteJoinsAsSemijoins: applying rewriting to join: Aggregate [s_store_name#2833], [s_store_name#2833 AS Store#3603, sum(ss_ext_sales_price#3477) AS Total_Sales#3604, (avg((ss_ext_discount_amt#3476 / cast(ss_quantity#3472 as decimal(10,0)))) * 100) AS Avg_Discount_Percentage#3605]
+- Project [ss_quantity#3472, ss_ext_discount_amt#3476, ss_ext_sales_price#3477, s_store_name#2833]
   +- Join Inner, (ss_store_sk#3469 = s_store_sk#2828)
      :- Project [ss_store_sk#3469, ss_quantity#3472, ss_ext_discount_amt#3476, ss_ext_sales_price#3477]
      :  +- Filter isnotnull(ss_store_sk#3469)
      :     +- Relation [ss_sold_date_sk#3462,ss_sold_time_sk#3463,ss_item_sk#3464,ss_customer_sk#3465,ss_cdemo_sk#3466,ss_hdemo_sk#3467,ss_addr_sk#3468,ss_store_sk#3469,ss_promo_sk#3470,ss_ticket_number#3471,ss_quantity#3472,ss_wholesale_cost#3473,ss_list_price#3474,ss_sales_price#3475,ss_ext_discount_amt#3476,ss_ext_sales_price#3477,ss_ext_wholesale_cost#3478,ss_ext_list_price#3479,

running query: 
SELECT

    ca_state AS State,

    SUM(cs_net_paid) AS Total_Spending,

    COUNT(DISTINCT c_customer_sk) AS Unique_Customers

FROM

    customer

JOIN

    customer_address

    ON customer.c_current_addr_sk = customer_address.ca_address_sk

JOIN

    catalog_sales

    ON customer.c_customer_sk = catalog_sales.cs_bill_customer_sk

GROUP BY

    ca_state

ORDER BY

    Total_Spending DESC;

timeout or error: An error occurred while calling o832.showString.
: java.lang.ClassCastException: class org.apache.spark.sql.catalyst.expressions.MakeDecimal cannot be cast to class org.apache.spark.sql.catalyst.expressions.aggregate.AggregateExpression (org.apache.spark.sql.catalyst.expressions.MakeDecimal and org.apache.spark.sql.catalyst.expressions.aggregate.AggregateExpression are in unnamed module of loader 'app')
	at scala.collection.TraversableLike.$anonfun$map$1(TraversableLike.scala:286)
	at scala.collection.LinearSeqOptimized.foreach(LinearSeqOptimized.scala:75)
	at sca

25/03/05 10:33:05 WARN RewriteJoinsAsSemijoins: applying rewriting to join: Aggregate [ca_state#2578], [ca_state#2578 AS State#3647, sum(cs_net_paid#3423) AS Total_Spending#3648, count(distinct c_customer_sk#2963) AS Unique_Customers#3649L]
+- Project [c_customer_sk#2963, ca_state#2578, cs_net_paid#3423]
   +- Join Inner, (c_customer_sk#2963 = cs_bill_customer_sk#3397)
      :- Project [c_customer_sk#2963, ca_state#2578]
      :  +- Join Inner, (c_current_addr_sk#2967 = ca_address_sk#2561)
      :     :- Project [c_customer_sk#2963, c_current_addr_sk#2967]
      :     :  +- Filter (isnotnull(c_current_addr_sk#2967) AND isnotnull(c_customer_sk#2963))
      :     :     +- Relation [c_customer_sk#2963,c_customer_id#2964,c_current_cdemo_sk#2965,c_current_hdemo_sk#2966,c_current_addr_sk#2967,c_first_shipto_date_sk#2968,c_first_sales_date_sk#2969,c_salutation#2970,c_first_name#2971,c_last_name#2972,c_preferred_cust_flag#2973,c_birth_day#2974,c_birth_month#2975,c_birth_year#2976,c_birth_count

running query: 
SELECT

    ca_state AS State,

    SUM(cs_ext_sales_price) AS Total_Spending,

    COUNT(DISTINCT c_customer_sk) AS Unique_Customers

FROM

    customer

JOIN

    customer_address

    ON customer.c_current_addr_sk = customer_address.ca_address_sk

JOIN

    catalog_sales

    ON customer.c_customer_sk = catalog_sales.cs_bill_customer_sk

GROUP BY

    ca_state

ORDER BY

    Total_Spending DESC;

timeout or error: An error occurred while calling o839.showString.
: java.lang.ClassCastException: class org.apache.spark.sql.catalyst.expressions.MakeDecimal cannot be cast to class org.apache.spark.sql.catalyst.expressions.aggregate.AggregateExpression (org.apache.spark.sql.catalyst.expressions.MakeDecimal and org.apache.spark.sql.catalyst.expressions.aggregate.AggregateExpression are in unnamed module of loader 'app')
	at scala.collection.TraversableLike.$anonfun$map$1(TraversableLike.scala:286)
	at scala.collection.LinearSeqOptimized.foreach(LinearSeqOptimized.scala:75)


25/03/05 10:33:05 WARN RewriteJoinsAsSemijoins: applying rewriting to join: Aggregate [ca_state#2578], [ca_state#2578 AS State#3700, sum(cs_ext_sales_price#3417) AS Total_Spending#3701, count(distinct c_customer_sk#2963) AS Unique_Customers#3702L]
+- Project [c_customer_sk#2963, ca_state#2578, cs_ext_sales_price#3417]
   +- Join Inner, (c_customer_sk#2963 = cs_bill_customer_sk#3397)
      :- Project [c_customer_sk#2963, ca_state#2578]
      :  +- Join Inner, (c_current_addr_sk#2967 = ca_address_sk#2561)
      :     :- Project [c_customer_sk#2963, c_current_addr_sk#2967]
      :     :  +- Filter (isnotnull(c_current_addr_sk#2967) AND isnotnull(c_customer_sk#2963))
      :     :     +- Relation [c_customer_sk#2963,c_customer_id#2964,c_current_cdemo_sk#2965,c_current_hdemo_sk#2966,c_current_addr_sk#2967,c_first_shipto_date_sk#2968,c_first_sales_date_sk#2969,c_salutation#2970,c_first_name#2971,c_last_name#2972,c_preferred_cust_flag#2973,c_birth_day#2974,c_birth_month#2975,c_birth_year#2976

## JOB (IMDB) Benchmark

In [ ]:
#### benchmark configuration
group_in_leaves = False
dbname = 'imdb'
runs = ['1', '2', '3', '4', '5', '6']
####

queries = ['job/2a.sql', 'job/2b.sql', 'job/2c.sql', 'job/2d.sql',
           'job/3a.sql', 'job/3b.sql', 'job/3c.sql',
           'job/5a.sql', 'job/5b.sql', 'job/5c.sql',
           'job/17a.sql', 'job/17b.sql', 'job/17c.sql', 'job/17d.sql', 'job/17e.sql', 'job/17f.sql',
           'job/20a.sql', 'job/20b.sql', 'job/20c.sql',
          ]

print('running queries: ' + str(queries))
benchmark_all(dbname, 'opt', runs, queries, physical_cj=True)
#benchmark_all(dbname, 'ref', runs, queries)

## STATS Benchmark

In [ ]:
#### benchmark configuration
dbname = 'stats'
#mode = 'opt'
runs = ['1', '2', '3', '4', '5', '6']
#runs = ['04']
#runs = ['01']
####

queries = sorted(glob.glob('stats-queries/*.sql'))
queries_hint = sorted(glob.glob('stats-queries/hints/*.sql'))

print('running queries: ' + str(queries))
#benchmark_all(dbname, 'opt', runs, queries, physical_cj=True)
benchmark_all(dbname, 'opt', runs, queries_hint, group_in_leaves=True, physical_cj=True)
#benchmark_all(dbname, 'ref', runs, queries)

## hetionet

In [ ]:
#### benchmark configuration
dbname = 'hetio'
#mode = 'opt'
runs = ['1', '2', '3', '4', '5', '6']
#runs = ['04']
#runs = ['01']
####

queries = sorted(glob.glob('hetio/*.sql'))
#queries = ['hetio/CtDpSpD.sql']

print('running queries: ' + str(queries))
#benchmark_all(dbname, 'opt', runs, queries, physical_cj=True)
benchmark_all(dbname, 'opt', runs, queries, group_in_leaves=False, physical_cj=True)
benchmark_all(dbname, 'opt', runs, queries, group_in_leaves=False, physical_cj=False)
benchmark_all(dbname, 'ref', runs, queries)
#benchmark_all(dbname, 'ref', runs, queries)

In [ ]:
spark = create_spark()
#import_db(spark, 'stats')
import_db(spark, 'lsqb')

In [ ]:
spark.sparkContext.setLogLevel("INFO")
spark.sql("SET spark.sql.yannakakis.enabled = false").show()
#spark.sql("SET spark.sql.yannakakis.enabled = false").show()
#df = run_query(spark, 'stats-queries/142-135.sql')
#df = run_query(spark, 'stats-queries/hints/142-135-hint.sql')
#df = run_query(spark, 'stats-queries/hints/141-068-hint.sql')
df = run_query(spark, 'lsqb/sql/q1.sql')
df.show()
print(explain_str(df))

In [ ]:

df = spark.sql('select count(*) from comments as c')
df.show()

In [ ]:
spark.sql('cache table comments').show()

In [ ]:
a1 = 'a'
a2 = 'a'

set([a1, 'b']) - set([a2])